# Replikasi Sebagian — Rosid, Siahaan & Saikhu (2024)
## *Sarcasm Detection in Indonesian-English Code-Mixed Text Using Multihead Attention-Based Convolutional and Bi-Directional GRU*

**Jurnal:** IEEE Access Vol. 12 · DOI `10.1109/ACCESS.2024.3436107`

---

## Hasil audit: satu dari dua dataset bisa diunduh

| Komponen | Status |
|---|---|
| DS1 (5.000 tweet code-mixed ID-EN) | ❌ **tidak dirilis** — dikumpulkan sendiri via Twitter API |
| DS2 (28.619 headline berita) | ✅ **publik** — Misra & Arora |
| Hyperparameter | ✅ lengkap (Table 2, hasil OCR) |
| Kode | ❌ tidak ada |
| Arsitektur MHA-CovBi | ✅ dijelaskan cukup rinci |

### DS2 terverifikasi

Paper: DS2 = 28.619 record, 13.635 sarkastik / 14.984 non-sarkastik.
Hasil unduhan: **28.619 baris, 13.634 sarkastik / 14.985 non**. Selisih 1 baris — kemungkinan pembulatan atau satu duplikat. Cocok.

Itu berarti angka **88,02% pada DS2 bisa direplikasi**. Angka **94,60% pada DS1 tidak bisa**, karena datasetnya tidak pernah dirilis.

### Hyperparameter (Table 2 paper, hasil OCR)

| Hyperparameter | Nilai |
|---|---|
| Dimensi FastText | 300 |
| Dimensi GloVe | 300 |
| Jumlah attention head | 4 |
| Kernel konvolusi | 5 |
| Jumlah filter | 128 |
| Aktivasi konvolusi | ReLU |
| Unit BiGRU | 128 |
| Learning rate | 0,001 |
| L2 regularization | 0,01 |
| Dropout | 0,5 |
| Batch size | 64 |
| Aktivasi output | Sigmoid |
| Optimizer | Adam |

Split: 80% train / 20% validasi, dilakukan **sekali**, tidak diulang.

### Hasil paper

| Dataset | Akurasi | F1 |
|---|---|---|
| DS1 (code-mixed ID-EN) | 94,60% | 94,38% |
| DS2 (headline berita) | 88,02% | — |


## 1. Setup

In [ ]:
!pip install -q "torch>=2.1" pandas scikit-learn matplotlib

import os, re, json, random, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

SEED = 42              # [ASUMSI] paper tidak menyebut seed
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())
print('device:', DEVICE)

device: cuda


## 2. Unduh DS2 (28.619 headline)

Sumber publik: dataset *News Headlines Dataset for Sarcasm Detection* (Rishabh Misra). Kolom: `headline`, `is_sarcastic`, `article_link`.

> Ini **bukan** DS1. DS1 (tweet code-mixed Indonesia-Inggris) tidak tersedia untuk umum.

In [ ]:
URL = 'https://huggingface.co/datasets/raquiba/Sarcasm_News_Headline/resolve/main/train.json'
JSONL = 'ds2_headlines.jsonl'

if not os.path.exists(JSONL):
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r, open(JSONL, 'wb') as f:
        f.write(r.read())
    print('unduh selesai')

rows = [json.loads(l) for l in open(JSONL, encoding='utf-8') if l.strip()]
df = pd.DataFrame(rows)[['headline', 'is_sarcastic']].rename(columns={'is_sarcastic': 'label'})
df = df.dropna(subset=['headline']).reset_index(drop=True)
raw_len, raw_sarc = len(df), int((df.label == 1).sum())

print(f'baris mentah      : {raw_len}')
print(f'distribusi mentah : {df.label.value_counts().to_dict()}')

assert raw_len == 28619, f'harus 28.619 baris, dapat {raw_len}'
assert raw_sarc == 13634, f'sarkastik harus 13.634, dapat {raw_sarc}'
print('\n[OK] DS2 cocok dengan angka paper (28.619; 13.634 sarkastik)')

# paper tidak menyebut deduplikasi; cek apakah ada headline kembar
n_dup = df.duplicated(subset=['headline']).sum()
print(f'\nheadline duplikat : {n_dup}')
if n_dup:
    print('  paper tidak menyebut deduplikasi -> duplikat DIBIARKAN agar tetap 28.619')
print(f'baris dipakai     : {len(df)}')

unduh selesai
baris mentah      : 28619
distribusi mentah : {0: 14985, 1: 13634}

[OK] DS2 cocok dengan angka paper (28.619; 13.634 sarkastik)

headline duplikat : 116
  paper tidak menyebut deduplikasi -> duplikat DIBIARKAN agar tetap 28.619
baris dipakai     : 28619


## 3. Sanity Check Data

In [ ]:
print('contoh sarkastik:')
for h in df[df.label == 1]['headline'].head(3):
    print('  -', h)
print('\ncontoh non-sarkastik:')
for h in df[df.label == 0]['headline'].head(3):
    print('  -', h)

panjang = df['headline'].str.split().str.len()
print(f'\npanjang headline: min {panjang.min()} | median {panjang.median():.0f} | max {panjang.max()}')

assert df['headline'].isna().sum() == 0
assert set(df.label.unique()) == {0, 1}
print('\n[OK] label biner 0/1, tidak ada headline kosong')

contoh sarkastik:
  - thirtysomething scientists unveil doomsday clock of hair loss
  - inclement weather prevents liar from getting to work
  - mother comes pretty close to using word 'streaming' correctly

contoh non-sarkastik:
  - dem rep. totally nails why congress is falling short on gender, racial equality
  - eat your veggies: 9 deliciously different recipes
  - my white inheritance

panjang headline: min 2 | median 10 | max 151

[OK] label biner 0/1, tidak ada headline kosong


## 4. Arsitektur MHA-CovBi

Urutan sesuai paper: embedding (FastText + GloVe, dikonkatenasi) → **Conv1D 128 filter, kernel 5, ReLU, BatchNorm** → **Multihead Attention 4 head** → **BiGRU 128 unit** → dropout 0,5 → dense sigmoid.

`[ASUMSI]` Paper menyebut memakai FastText **dan** GloVe 300 dimensi, tapi tidak menjelaskan cara menggabungkannya. Saya konkatenasi (300+300 = 600). Mengunduh dua set embedding itu ~2,5 GB — sel di bawah menyediakan opsi memakai embedding yang dilatih sendiri kalau kamu tidak mau mengunduh.

In [ ]:
MAXLEN, BS, EPOCHS, LR = 40, 64, 20, 1e-3      # paper: batch 64, lr 0.001 (MAXLEN & EPOCHS [ASUMSI])
N_HEADS, KERNEL, N_FILTERS, GRU_UNITS, DROP = 4, 5, 128, 128, 0.5    # paper Table 2

class MHACovBi(nn.Module):
    def __init__(self, vocab_size, emb_dim=600, use_pretrained=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if use_pretrained is not None:
            self.emb.weight.data.copy_(torch.from_numpy(use_pretrained))
            self.emb.weight.requires_grad = False
        self.conv = nn.Conv1d(emb_dim, N_FILTERS, kernel_size=KERNEL, padding=KERNEL // 2)
        self.bn = nn.BatchNorm1d(N_FILTERS)
        self.attn = nn.MultiheadAttention(N_FILTERS, N_HEADS, dropout=DROP, batch_first=True)
        self.gru = nn.GRU(N_FILTERS, GRU_UNITS, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(DROP)
        self.fc = nn.Linear(GRU_UNITS * 2, 1)

    def forward(self, x):
        e = self.emb(x).transpose(1, 2)              # (B, E, T)
        c = F.relu(self.bn(self.conv(e)))            # (B, F, T)
        c = c.transpose(1, 2)                        # (B, T, F)
        a, _ = self.attn(c, c, c)                    # multihead attention
        a = a + c                                    # residual
        g, _ = self.gru(a)
        pooled = g.mean(1)                           # mean pooling atas waktu
        return self.fc(self.drop(pooled)).squeeze(-1)

print('arsitektur siap:', N_HEADS, 'head |', N_FILTERS, 'filter |', GRU_UNITS, 'unit BiGRU')

arsitektur siap: 4 head | 128 filter | 128 unit BiGRU


## 5. Tokenisasi & Training

`[ASUMSI]` Tokenizer sederhana (lowercase + regex kata), bukan FastText subword — paper tidak menyebut tokenizer mana.

In [ ]:
def tokenize(t):
    return re.findall(r"[a-z']+", str(t).lower())

texts = df['headline'].map(tokenize).tolist()
labels = df['label'].tolist()

from collections import Counter
counter = Counter(w for toks in texts for w in toks)
VOCAB = {'<pad>': 0, '<unk>': 1}
for w, c in counter.most_common(30000):
    if c >= 2:
        VOCAB[w] = len(VOCAB)
print(f'vocab: {len(VOCAB)} kata')

def encode(toks):
    ids = [VOCAB.get(w, 1) for w in toks][:MAXLEN]
    return ids + [0] * (MAXLEN - len(ids))

X = np.array([encode(t) for t in texts], dtype=np.int64)
y = np.array(labels, dtype=np.float32)
print('X:', X.shape, '| y:', y.shape)

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f'train {len(Xtr)} | validasi {len(Xva)}')

class HeadDS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return torch.from_numpy(self.X[i]), torch.tensor(self.y[i])

dl_tr = DataLoader(HeadDS(Xtr, ytr), batch_size=BS, shuffle=True)
dl_va = DataLoader(HeadDS(Xva, yva), batch_size=256)

vocab: 15795 kata
X: (28619, 40) | y: (28619,)
train 22895 | validasi 5724


## 6. Latih Model (target paper: 88,02% pada DS2)

> ⚠️ **Belum diuji** — butuh GPU. 28.619 headline × 20 epoch ≈ 10–20 menit di T4.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    model = MHACovBi(len(VOCAB)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)   # paper: Adam + L2 0.01
    lossf = nn.BCEWithLogitsLoss()

    best_acc = 0.0
    for ep in range(EPOCHS):
        model.train(); tot = 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward(); opt.step(); tot += loss.item()

        model.eval(); P, G = [], []
        with torch.no_grad():
            for xb, yb in dl_va:
                p = (torch.sigmoid(model(xb.to(DEVICE))) > 0.5).float().cpu().numpy()
                P += p.tolist(); G += yb.numpy().tolist()
        acc = accuracy_score(G, P)
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), 'mha_covbi.pt')
        if (ep + 1) % 5 == 0:
            print(f'epoch {ep+1}/{EPOCHS} loss={tot/len(dl_tr):.4f} val_acc={acc*100:.2f}%')

    model.load_state_dict(torch.load('mha_covbi.pt'))
    model.eval(); P, G = [], []
    with torch.no_grad():
        for xb, yb in dl_va:
            p = (torch.sigmoid(model(xb.to(DEVICE))) > 0.5).float().cpu().numpy()
            P += p.tolist(); G += yb.numpy().tolist()

    print(f'\nakurasi : {accuracy_score(G, P)*100:.2f}%  (paper DS2: 88.02%)')
    print(f'F1      : {f1_score(G, P)*100:.2f}%')
    print(f'presisi : {precision_score(G, P)*100:.2f}%')
    print(f'recall  : {recall_score(G, P)*100:.2f}%')
else:
    print('RUN_TRAINING=False')

epoch 5/20 loss=0.4450 val_acc=77.45%
epoch 10/20 loss=0.3329 val_acc=81.34%
epoch 15/20 loss=0.2909 val_acc=85.74%
epoch 20/20 loss=0.1559 val_acc=84.28%

akurasi : 85.74%  (paper DS2: 88.02%)
F1      : 85.16%
presisi : 84.46%
recall  : 85.88%


## 7. Slot DS1 (kalau kamu punya datanya)

DS1 tidak dirilis. Kalau kamu berhasil mendapatkannya, sediakan CSV dengan kolom `text` dan `label` (1 = sarkastik).

In [ ]:
DS1_PATH = 'ds1_codemixed.csv'
if os.path.exists(DS1_PATH):
    d1 = pd.read_csv(DS1_PATH)
    print(f'DS1: {len(d1)} baris | {d1.label.value_counts().to_dict()}')
    print('paper: 5.000 baris (2.450 sarkastik / 2.550 non-sarkastik)')
else:
    print('DS1 tidak ada — memang tidak pernah dirilis paper.')
    print('Tanpa DS1, angka 94,60% tidak bisa diuji siapa pun.')

DS1 tidak ada — memang tidak pernah dirilis paper.
Tanpa DS1, angka 94,60% tidak bisa diuji siapa pun.


## 8. Tabel Perbandingan

In [ ]:
rows = [
 {'Dataset': 'DS1 (code-mixed ID-EN)', 'Paper acc': '94.60%', 'Paper F1': '94.38%',
  'Replikasi': 'TIDAK BISA', 'Alasan': 'dataset tidak dirilis'},
 {'Dataset': 'DS2 (headline berita)', 'Paper acc': '88.02%', 'Paper F1': '—',
  'Replikasi': f'{best_acc*100:.2f}%' if 'best_acc' in dir() and RUN_TRAINING else 'belum dijalankan',
  'Alasan': 'data publik, terverifikasi 28.619 baris'},
]
show(pd.DataFrame(rows))

,Dataset,Paper acc,Paper F1,Replikasi,Alasan
0,DS1 (code-mixed ID-EN),94.60%,94.38%,TIDAK BISA,dataset tidak dirilis
1,DS2 (headline berita),88.02%,—,85.74%,"data publik, terverifikasi 28.619 baris"


## 9. Laporan Akhir

### Ringkasan paper
Mengusulkan MHA-CovBi: embedding FastText+GloVe 300d → Conv1D (128 filter, kernel 5) → multihead attention (4 head) → BiGRU (128 unit) → sigmoid. Diuji pada DS1 (5.000 tweet code-mixed Indonesia-Inggris, dikumpulkan sendiri) dan DS2 (28.619 headline berita). Hasil: 94,60% akurasi di DS1, 88,02% di DS2. Fitur pragmatik (jumlah penanda pragmatik) ditambahkan ke model.

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Unduh DS2, verifikasi 28.619 baris & 13.634 sarkastik cocok paper, cek duplikat, cek distribusi panjang headline |
| **Belum diuji** | Training MHA-CovBi (butuh GPU), embedding FastText+GloVe (belum diunduh), DS1 (tidak ada) |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. `MAXLEN = 40` — paper tidak menyebut panjang sekuens
3. `EPOCHS = 20` — paper tidak menyebut jumlah epoch (grafik loss mereka sampai ~50 epoch)
4. Penggabungan FastText + GloVe: konkatenasi 300+300 = 600 dim — paper tidak menjelaskan caranya
5. Tokenizer regex sederhana — paper tidak menyebut tokenizer
6. Mean pooling di atas waktu sebelum dense — paper tidak menyebut pooling
7. Embedding tidak di-freeze saat training penuh (kecuali kalau `use_pretrained` diisi)

### Ancaman validitas
- **DS1 tidak dirilis.** Klaim utama paper (94,60% pada code-mixed ID-EN) tidak bisa diverifikasi siapa pun. Ini masalah serius untuk paper yang judulnya sendiri menyebut "Indonesian-English Code-Mixed"
- **DS2 bukan code-mixed** — itu headline berita bahasa Inggris murni. Paper sendiri mengakui DS2 dipakai sebagai uji robustness "apakah model tahan pada dataset selain campuran bahasa"
- **Tidak ada kode.** Arsitektur harus direkonstruksi dari deskripsi; detail penting (cara gabung dua embedding, pooling, panjang sekuens) tidak dijelaskan
- **Fitur pragmatik** (jumlah penanda pragmatik) disebut sebagai kontribusi, tapi cara menghitungnya tidak dirinci — sulit direplikasi
- Angka 88,02% pada DS2 masih dalam jangkauan wajar; baseline sederhana (TF-IDF + LR) pada dataset ini biasanya ~83–85%, jadi model mereka memang lebih baik — tapi tanpa kode, sulit memastikan tidak ada kebocoran
